In [1]:
import google.generativeai as genai
import os
import re
import time
import numpy as np # NumPyを分散計算に使用

# --- Configuration ---
API_KEY_FILE = "apikey.txt"
MODEL_NAME = 'gemini-2.5-flash-preview-05-20' # 最新のモデル、または 'gemini-pro'
# MODEL_NAME = 'gemini-pro'
NUM_EVALUATIONS_PER_SENRYU = 30 # 各川柳に対する評価回数
REQUEST_DELAY_SECONDS = 1 # APIリクエスト間の遅延（秒）レート制限対策

def get_api_key(filepath=API_KEY_FILE):
    """指定されたファイルからAPIキーを読み込む"""
    try:
        with open(filepath, "r") as f:
            return f.read().strip()
    except FileNotFoundError:
        print(f"エラー: APIキーファイル '{filepath}' が見つかりません。")
        return None

def extract_evaluation_score(evaluation_text, scale=10):
    """
    LLMの応答テキストから評価点 (数値) を抽出する試み。
    例: "評価点: 7/10", "7点", "スコアは7です", "7"
    """
    if not evaluation_text:
        return None

    # "評価点: 7/10" or "7 / 10"
    match = re.search(r"(\d+)\s*/\s*\d+", evaluation_text)
    if match:
        try:
            score = int(match.group(1))
            if 0 <= score <= scale:
                return score
        except ValueError:
            pass # 数値変換失敗

    # "評価点: 7", "スコア:7", "7点" (数字が先頭に来る場合も考慮)
    # "〇点" という形式を優先的に探す
    match = re.search(r"(\d+)\s*(?:点|ポイント)", evaluation_text)
    if match:
        try:
            score = int(match.group(1))
            if 0 <= score <= scale:
                return score
        except ValueError:
            pass

    # "評価点<記号><数字>", "スコア<記号><数字>"
    match = re.search(r"(?:評価点|スコア|点数)\s*[:：=\-]\s*(\d+)", evaluation_text, re.IGNORECASE)
    if match:
        try:
            score = int(match.group(1))
            if 0 <= score <= scale:
                return score
        except ValueError:
            pass

    # テキスト中の単独の数字で、スケール内のものを探す (最後の手段)
    # 応答が数字のみの場合や、"Result: 7" のような場合を想定
    numbers_found = re.findall(r"\b(\d+)\b", evaluation_text) # 単語境界で区切られた数字
    for num_str in numbers_found:
        try:
            score = int(num_str)
            if 0 <= score <= scale:
                # これだけだと他の数字（例：お題番号など）を誤検出する可能性あり
                # より確実性を高めるならプロンプトで回答形式を厳密に指定する
                return score # 最初に見つかった妥当な範囲の数値を返す
        except ValueError:
            continue
    return None

def get_single_llm_evaluation(model, senryu_to_evaluate, theme, scale, context_info=""):
    """
    LLMに単一の川柳を評価させ、テキスト応答を返す。
    """
    prompt = f"""あなたは川柳の評価者です。
以下の川柳について、その「面白さ」を1から{scale}の{scale}段階で評価してください。
({scale}が最も面白いとします)。

お題: 「{theme}」
{context_info}

評価対象の川柳:
「{senryu_to_evaluate}」

回答形式の希望:
評価点をまず数字で示し (例: 7)、その後に評価理由を続けてください。
例1: 7
理由: [ここに具体的な理由]
例2: 8点。この句はユーモラスで良い。

評価点のみを抽出するため、点数は明確に示してください。
"""
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        # print(f"  LLM評価API呼び出しエラー: {e}") # 詳細なエラーは呼び出し側で制御
        raise e # エラーを再スローして呼び出し側で処理

if __name__ == "__main__":
    api_key = get_api_key()

    senryu_collection_to_test = [
        "赤本を 抱いて挑むは 国立大",
        "国立の 桜トンネル 春を呼ぶ",
        "歓声が 響く国立 夢舞台",
        "学園の 香り漂う 国立路",
        "太古から 国立展示 知の宝",
        "国の施設 予算は常に 大規模だ",
        "大学の 通り散歩で 昼下がり",
        "広大に 緑あふれる 国立園",
        "銀杏散る 黄色い絨毯 くにたちの",
        "安い学費 親もニッコリ 国立だ"
    ]

    if api_key:
        genai.configure(api_key=api_key)
        try:
            model = genai.GenerativeModel(MODEL_NAME)
            print(f"使用モデル: {MODEL_NAME}")
        except Exception as e:
            print(f"モデルの初期化中にエラーが発生しました ({MODEL_NAME}): {e}")
            exit()

        theme_of_senryu = "国立"
        evaluation_scale = 10
        location_context_info = f"""
評価の際のヒント:
これらの川柳は「{theme_of_senryu}」というお題で作成されました。
「国立」には、国の施設や機関（例：国立競技場、国立大学、国立博物館、国立公園など）を指す「こくりつ」と、
地名である東京都「国立市（くにたちし）」（学園都市、大学通り、桜並木などが知られる）を指す「くにたち」の二つの意味合いがあります。
この背景情報を考慮して、「面白さ」の観点から評価をお願いします。
"""

        print(f"\nお題「{theme_of_senryu}」の川柳について、LLMによる評価の頑健性調査を開始します。")
        print(f"各川柳に対して{NUM_EVALUATIONS_PER_SENRYU}回の評価を行い、評価点の分散を計算します。")
        print(f"各評価リクエスト間に{REQUEST_DELAY_SECONDS}秒の遅延を設けます。")
        print(f"総API呼び出し回数: {len(senryu_collection_to_test) * NUM_EVALUATIONS_PER_SENRYU}回")
        estimated_time_minutes = (len(senryu_collection_to_test) * NUM_EVALUATIONS_PER_SENRYU * REQUEST_DELAY_SECONDS) / 60
        print(f"推定所要時間: 約{estimated_time_minutes:.1f}分 (処理時間含まず)\n")

        all_results = {}

        for i, senryu in enumerate(senryu_collection_to_test):
            # 川柳本文から先頭の番号や不要な文字を除去（今回はユーザー提供リストがクリーンなので不要）
            # cleaned_senryu = re.sub(r'^[0-9０-９]+[\.．、\s]*', '', senryu).strip()
            cleaned_senryu = senryu

            print(f"--- 川柳 No.{i + 1}: 「{cleaned_senryu}」の評価 ({NUM_EVALUATIONS_PER_SENRYU}回) を開始 ---")
            scores_for_this_senryu = []
            successful_eval_count = 0

            for j in range(NUM_EVALUATIONS_PER_SENRYU):
                print(f"  試行 {j + 1}/{NUM_EVALUATIONS_PER_SENRYU}...")
                try:
                    evaluation_text = get_single_llm_evaluation(
                        model,
                        cleaned_senryu,
                        theme_of_senryu,
                        evaluation_scale,
                        location_context_info
                    )
                    score = extract_evaluation_score(evaluation_text, evaluation_scale)

                    if score is not None:
                        scores_for_this_senryu.append(score)
                        successful_eval_count += 1
                        print(f"    -> 抽出スコア: {score}")
                    else:
                        print(f"    -> スコア抽出失敗。応答抜粋: \"{evaluation_text[:100].replace(chr(10), ' ')}...\"")

                except Exception as e:
                    print(f"    -> API呼び出し中にエラーが発生: {e}")
                    # エラーによってはリトライ処理を入れることも検討できる
                
                if j < NUM_EVALUATIONS_PER_SENRYU - 1: # 最後の呼び出しの後には待機しない
                    time.sleep(REQUEST_DELAY_SECONDS)

            variance = np.var(scores_for_this_senryu) if successful_eval_count > 1 else 0.0
            mean_score = np.mean(scores_for_this_senryu) if successful_eval_count > 0 else float('nan')
            
            all_results[cleaned_senryu] = {
                'scores': scores_for_this_senryu,
                'variance': variance,
                'mean': mean_score,
                'successful_evals': successful_eval_count,
                'total_attempts': NUM_EVALUATIONS_PER_SENRYU
            }

            print(f"  評価完了: 成功{successful_eval_count}/{NUM_EVALUATIONS_PER_SENRYU}回, 平均点={mean_score:.2f}, 分散={variance:.4f}")
            print("-----------------------------------------------------\n")

        print("\n--- 全川柳の評価頑健性調査結果まとめ ---")
        for senryu, data in all_results.items():
            print(f"川柳: 「{senryu}」")
            print(f"  成功評価/試行回数: {data['successful_evals']}/{data['total_attempts']}")
            if data['successful_evals'] > 0:
                print(f"  評価点 (一部): {data['scores'][:min(5, len(data['scores']))]}...") # 最初の最大5件
                print(f"  平均評価点: {data['mean']:.2f}")
                print(f"  評価点の分散: {data['variance']:.4f}")
            else:
                print(f"  有効な評価スコアを取得できませんでした。")
            print("-" * 20)
    else:
        print("APIキーがないため、プログラムを実行できません。")

c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


使用モデル: gemini-2.5-flash-preview-05-20

お題「国立」の川柳について、LLMによる評価の頑健性調査を開始します。
各川柳に対して30回の評価を行い、評価点の分散を計算します。
各評価リクエスト間に1秒の遅延を設けます。
総API呼び出し回数: 300回
推定所要時間: 約5.0分 (処理時間含まず)

--- 川柳 No.1: 「赤本を 抱いて挑むは 国立大」の評価 (30回) を開始 ---
  試行 1/30...
    -> 抽出スコア: 5
  試行 2/30...
    -> 抽出スコア: 6
  試行 3/30...
    -> 抽出スコア: 4
  試行 4/30...
    -> 抽出スコア: 5
  試行 5/30...
    -> 抽出スコア: 5
  試行 6/30...
    -> 抽出スコア: 6
  試行 7/30...
    -> 抽出スコア: 7
  試行 8/30...
    -> 抽出スコア: 4
  試行 9/30...
    -> 抽出スコア: 5
  試行 10/30...
    -> 抽出スコア: 5
  試行 11/30...
    -> 抽出スコア: 5
  試行 12/30...
    -> 抽出スコア: 7
  試行 13/30...
    -> 抽出スコア: 5
  試行 14/30...
    -> 抽出スコア: 5
  試行 15/30...
    -> 抽出スコア: 5
  試行 16/30...
    -> 抽出スコア: 5
  試行 17/30...
    -> 抽出スコア: 6
  試行 18/30...
    -> 抽出スコア: 5
  試行 19/30...
    -> 抽出スコア: 5
  試行 20/30...
    -> 抽出スコア: 5
  試行 21/30...
    -> 抽出スコア: 5
  試行 22/30...
    -> 抽出スコア: 4
  試行 23/30...
    -> 抽出スコア: 7
  試行 24/30...
    -> 抽出スコア: 5
  試行 25/30...
    -> 抽出スコア: 2
  試行 26/30...
    -> 抽出スコア: 5
  試行 27/30..

KeyboardInterrupt: 